<a href="https://colab.research.google.com/github/jcdevaney/voiceAndGender/blob/main/notebooks/Audio_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import libraries and data

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import scipy.signal as sps
!pip install praat-parselmouth
import parselmouth

!git clone https://github.com/jcdevaney/voiceAndGender.git
# Define audio folder path
folder_path = '/content/voiceAndGender/data/musicLMsubset'

# Define parameters
hop_len = 512
nfft = 2048
min_pitchVal = librosa.note_to_hz('C1') # Minimum pitch value
max_pitchVal = librosa.note_to_hz('C7') # Maximum pitch value

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 105.8 MB/s eta 0:00:00
Cloning into 'voiceAndGender'...
remote: Enumerating objects: 2127, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 2127 (delta 9), reused 16 (delta 5), pack-reused 2105 (from 3)
Receiving objects: 100% (2127/2127), 774.25 MiB | 46.78 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Updating files: 100% (2093/2093), done.


## Vocal Presence Detection Function

In [2]:
# Some of the code in this cell was written with assistance of ChatGPT and Gemini
def vocal_presence(y, sr, nfft=1024, hop_len=256):
    # Features: Root Mean Square Energy (RMSE) and Zero Crossing Rate (ZCR)
    rmse = librosa.feature.rms(y=y, frame_length=nfft, hop_length=hop_len, center=False)[0]
    zcr  = librosa.feature.zero_crossing_rate(y=y, frame_length=nfft, hop_length=hop_len, center=False)[0]

    # Clip-adaptive thresholds (percentile-based) + sanity clamps for robustness
    # Energy (RMS) thresholds
    ste_on  = max(np.percentile(rmse, 65), 0.02)   # Activation threshold for RMS (don't go below 0.02 on normalized audio)
    ste_off = max(np.percentile(rmse, 45), 0.015)  # Deactivation threshold for RMS

    # ZCR thresholds (lower ZCR typically indicates more voiced sound) – invert logic with hysteresis
    zcr_on  = np.clip(np.percentile(zcr, 35), 0.08, 0.18) # Activation threshold for ZCR
    zcr_off = np.clip(np.percentile(zcr, 65), 0.18, 0.30) # Deactivation threshold for ZCR

    # Hysteresis state machine to determine vocal presence frames
    is_vocal = np.zeros_like(rmse, dtype=bool)
    state = False
    for i, (e, z) in enumerate(zip(rmse, zcr)):
        if not state:
            state = (e >= ste_on) and (z <= zcr_on) # Activate if energy is high and ZCR is low
        else:
            state = not ((e <= ste_off) or (z >= zcr_off)) # Deactivate if energy is low or ZCR is high
        is_vocal[i] = state

    # Debounce flicker: Apply a median filter to smooth out short, transient vocal/non-vocal segments
    # Kernel size 'k' is calculated to be approximately 30-50 ms
    k = int(round(0.04 * sr / hop_len)) | 1  # Ensure odd kernel size, ~40 ms
    is_vocal = sps.medfilt(is_vocal.astype(float), kernel_size=max(k, 3)) > 0.5
    return is_vocal

## Feature Extraction Function


In [3]:
# Some of the code in this cell was written with assistance of ChatGPT and Gemini

def extract_features(audio_path):
    y, sr = librosa.load(audio_path)
    y = y[:sr*10]

    window_length = nfft / sr
    periods_per_window = window_length * min_pitchVal
    ts = hop_len / sr

    # calculate features
    f0, _, _ = librosa.pyin(y, fmin=min_pitchVal, fmax=max_pitchVal, frame_length=nfft, hop_length=hop_len, center=False)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=nfft, hop_length=hop_len, center=False)
    S, phase = librosa.magphase(librosa.stft(y, n_fft=nfft, hop_length=hop_len, center=False))
    rms = librosa.feature.rms(S=S)

    sound = parselmouth.Sound(audio_path)
    harm = parselmouth.Sound.to_harmonicity_cc(sound, time_step=ts, minimum_pitch=min_pitchVal, periods_per_window=periods_per_window)
    hnr = harm.values.flatten()

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=nfft, hop_length=hop_len, center=False)

    # run vocal presence detector
    vp = vocal_presence(y, sr, nfft, hop_len)

    # Reduce features only for frames where vocal presence is detected
    f0_frames = f0[vp]
    spectral_centroid_frames = spectral_centroid.flatten()[vp]
    rms_frames = rms.flatten()[vp]

    # Align hnr with vocal presence mask and then apply mask
    if len(hnr) > len(vp):
        hnr_aligned = hnr[:len(vp)]
    elif len(hnr) < len(vp):
        hnr_aligned = np.pad(hnr, (0, len(vp) - len(hnr)), 'constant', constant_values=np.nan)
    else:
        hnr_aligned = hnr
    hnr_frames = hnr_aligned[vp]

    # Create a common mask for all features to ensure alignment and remove NaNs
    common_valid_mask = ~np.isnan(f0_frames) & ~np.isnan(spectral_centroid_frames) & ~np.isnan(rms_frames) & ~np.isnan(hnr_frames)

    f0_clean = f0_frames[common_valid_mask]
    rms_clean = rms_frames[common_valid_mask]
    spectral_centroid_clean = spectral_centroid_frames[common_valid_mask]
    hnr_clean = hnr_frames[common_valid_mask]

    mfccs = mfccs[1:13, vp][:, common_valid_mask]

    # Calculate deltas on the cleaned arrays for F0 and RMS
    f0_delta = np.array([])
    if f0_clean.size >= 9:
        try:
            f0_delta = librosa.feature.delta(f0_clean)
        except ValueError:
            f0_delta = np.array([])

    rms_delta = np.array([])
    if rms_clean.size >= 9:
        try:
            rms_delta = librosa.feature.delta(rms_clean)
        except ValueError:
            rms_delta = np.array([])

    # Calculate means and percentiles
    fmin = np.min(f0_clean) if f0_clean.size > 0 else None
    fmax = np.max(f0_clean) if f0_clean.size > 0 else None
    f0_quartiles = np.percentile(f0_clean, [25, 50, 75]) if f0_clean.size > 0 else [None, None, None]

    f0_mean = np.nanmean(f0_clean) if f0_clean.size > 0 else None
    f0_delta_mean = np.nanmean(f0_delta) if f0_delta.size > 0 else None
    rms_mean = np.nanmean(rms_clean) if rms_clean.size > 0 else None
    rms_delta_mean = np.nanmean(rms_delta) if rms_delta.size > 0 else None
    spectral_centroid_mean = np.nanmean(spectral_centroid_clean) if spectral_centroid_clean.size > 0 else None

    adjusted_centroid = np.divide(spectral_centroid_clean, f0_clean, where=f0_clean != 0)
    adjusted_centroid = np.nan_to_num(adjusted_centroid, nan=0.0)
    adjusted_centroid_mean = np.mean(adjusted_centroid) if adjusted_centroid.size > 0 else None

    hnr_mean = np.mean(hnr_clean) if hnr_clean.size > 0 else None

    mfcc_means = np.mean(mfccs, axis=1).tolist() if mfccs.shape[1] > 0 else [np.nan] * 12

    return (fmin, fmax, *f0_quartiles, f0_mean, f0_delta_mean, rms_mean, rms_delta_mean,
            spectral_centroid_mean, adjusted_centroid_mean, hnr_mean,
            *mfcc_means)

## Estimate Features on Audio Clips and Save to CSV File

In [4]:
# Some of the code in this cell was written with assistance of ChatGPT and Gemini

# Find all audio files in folder_path folder
audio_files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]

# Collect results for all audio files
results = [[f, *extract_features(os.path.join(folder_path, f))] for f in audio_files]

# Create a DataFrame for the new features with 'name' as the first column
new_data = pd.DataFrame(results, columns=['name', 'fmin (Hz)', 'fmax (Hz)', 'F0 25th percentile (Hz)',
                                          'F0 50th percentile (Hz)', 'F0 75th percentile (Hz)', 'F0 Mean (Hz)',
                                          'F0 Delta Mean (Hz)', 'RMS Mean', 'RMS Delta Mean',
                                          'Spectral Centroid Mean (Hz)',
                                          'Adjusted Centroid Mean', 'HNR Mean']
                                          + [f'MFCC {i} Mean' for i in range(2, 14)])

# Save the new DataFrame to a CSV file
new_csv_path = os.path.join(folder_path, 'AudioFeatures.csv')
new_data.to_csv(new_csv_path, index=False)